# Descarga de datos para entrenar el DETECTOR v3

Para entrenar `cnn_damage_detection_training3` en una computadora que no tiene las imagenes.
Se corre **una sola vez**, antes del entrenamiento.

Requisitos: AWS CLI instalado y configurado (`aws configure`) con acceso de lectura al bucket.

Baja dos cosas, porque el detector usa las dos:
- `data/damaged/`, `data/undamaged/` y `data/undamaged_superficies/`: sus fotos propias
- `data_clasificacion/`: las fotos de clasificacion, de donde toma una parte como ejemplos "con dano"

Todo queda en `notebooks/` (un nivel arriba de esta carpeta), en carpetas que estan en `.gitignore`.
Nada de lo que hace este notebook modifica el bucket.

In [ ]:
# Celda 1: configuracion

import os
import subprocess

BUCKET = "s3://humanoid-damage-cnn-834257582282-us-east-1-an/processed"
DATA_DETECCION = "../data"  # los datos quedan en notebooks/, fuera de parte3

In [ ]:
# Celda 2: imagenes propias del detector
# Baja damaged/, undamaged/ y undamaged_superficies/. "sync" solo copia lo
# que falta, asi que si se corta se puede volver a correr.

for carpeta in ["damaged", "undamaged", "undamaged_superficies"]:
    destino = f"{DATA_DETECCION}/{carpeta}"
    os.makedirs(destino, exist_ok=True)
    subprocess.run(["aws", "s3", "sync", f"{BUCKET}/damage_detection/{carpeta}/", f"{destino}/", "--quiet"], check=True)
    total = sum(len(archivos) for _, _, archivos in os.walk(destino))
    print(f"{destino}: {total} archivos")

In [ ]:
# Celda 3: imagenes de clasificacion y dataset_final.csv
# Corre el notebook que ya arma el dataset de clasificacion: baja las
# imagenes y etiquetas de S3, aplica los 11 grupos finales, limpia
# duplicados y archivos rotos, y guarda data_clasificacion/dataset_final.csv
# y grupos_final.json.
# Se corre desde notebooks/ porque ese notebook usa rutas relativas a esa carpeta.

os.chdir("..")
try:
    %run cnn_damage_classification_build_dataset.ipynb
finally:
    os.chdir("parte3")

In [ ]:
# Celda 4: control final

listo = all([
    os.path.exists("../data_clasificacion/dataset_final.csv"),
    *[bool(os.listdir(f"{DATA_DETECCION}/{c}")) for c in ["damaged", "undamaged", "undamaged_superficies"]],
])
print("Todo listo para entrenar el detector v3" if listo else "Falta algo: revisar las celdas anteriores")